# Project Work part 2 - Data Sources


Name: Alina Shah  
GitHub: https://github.com/59762/IND320-59762  
Streamlit: https://ind320-59762.streamlit.app/

**Setup and credentials**

The ENTSO-E token and the MongoDB connection string are read from `.streamlit/secrets.toml` (the same file the Streamlit app uses locally), with environment variables as a fallback. The secrets are never printed, and the file is listed in `.gitignore` so it is not pushed to GitHub.

Run this notebook from the root of the repository, so the relative path to `.streamlit/secrets.toml` works.

In [ ]:
# Standard library
import os
import re
import time
import tomllib                      # reads secrets.toml (Python 3.11+)
from datetime import datetime, timedelta, timezone
from pathlib import Path
import xml.etree.ElementTree as ET

# Third-party
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)


def load_secrets(path=".streamlit/secrets.toml"):
    # Read the secrets file if it exists, otherwise return an empty dict
    p = Path(path)
    if p.exists():
        with open(p, "rb") as f:
            return tomllib.load(f)
    return {}


SECRETS = load_secrets()
ENTSOE_TOKEN = SECRETS.get("entsoe_token") or os.getenv("ENTSOE_TOKEN")
MONGODB_URI = SECRETS.get("mongodb_uri") or os.getenv("MONGODB_URI")

# Only report whether the secrets were found - never print the values
print("ENTSO-E token found:", bool(ENTSOE_TOKEN))
print("MongoDB URI found:  ", bool(MONGODB_URI))
assert ENTSOE_TOKEN and MONGODB_URI, "Add entsoe_token and mongodb_uri to .streamlit/secrets.toml"

Cassandra runs locally and is used as the raw data store for the API data. If Cassandra is not running yet, it can for example be started with Docker:

`docker run --name cassandra -p 9042:9042 -d cassandra:latest`

In [ ]:
import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql import types as T

CASSANDRA_HOST = "127.0.0.1"
CASSANDRA_PORT = 9042
KEYSPACE = "ind320"
TABLE = "entsoe_flows"

# Spark Cassandra Connector version must match the Spark version
CONNECTOR_BY_SPARK = {
    "3.3": "com.datastax.spark:spark-cassandra-connector_2.12:3.3.0",
    "3.4": "com.datastax.spark:spark-cassandra-connector_2.12:3.4.1",
    "3.5": "com.datastax.spark:spark-cassandra-connector_2.12:3.5.1",
}
spark_minor = ".".join(pyspark.__version__.split(".")[:2])
CONNECTOR = CONNECTOR_BY_SPARK.get(spark_minor)
if CONNECTOR is None:
    raise RuntimeError(
        f"No connector mapped for Spark {pyspark.__version__}. "
        "Set CONNECTOR manually to a matching spark-cassandra-connector package."
    )

spark = (
    SparkSession.builder
    .appName("IND320-part2")
    .master("local[*]")
    .config("spark.jars.packages", CONNECTOR)
    .config("spark.cassandra.connection.host", CASSANDRA_HOST)
    .config("spark.cassandra.connection.port", str(CASSANDRA_PORT))
    .config("spark.sql.session.timeZone", "UTC")      # keep everything in UTC
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version, "| connector:", CONNECTOR)

In [ ]:
# Test 1: connect to Cassandra with the Python driver and create keyspace + table
from cassandra.cluster import Cluster

cluster = Cluster([CASSANDRA_HOST], port=CASSANDRA_PORT)
cass_session = cluster.connect()
print("Cassandra version:", cass_session.execute("SELECT release_version FROM system.local").one()[0])

cass_session.execute(f'''
    CREATE KEYSPACE IF NOT EXISTS {KEYSPACE}
    WITH replication = {{'class': 'SimpleStrategy', 'replication_factor': 1}}
''')

# One partition per connection + direction + year; rows sorted by time inside the partition.
# Re-inserting the same (corridor, direction, year, ts) overwrites the row, so re-running is safe.
cass_session.execute(f'''
    CREATE TABLE IF NOT EXISTS {KEYSPACE}.{TABLE} (
        corridor       text,
        direction      text,
        year           int,
        ts             timestamp,
        flow_mw        double,
        resolution_min int,
        out_zone       text,
        in_zone        text,
        PRIMARY KEY ((corridor, direction, year), ts)
    ) WITH CLUSTERING ORDER BY (ts ASC)
''')
print(f"Keyspace '{KEYSPACE}' and table '{TABLE}' are ready")

In [ ]:
# Test 2: check that Spark can read from Cassandra through the connector
(
    spark.read
    .format("org.apache.spark.sql.cassandra")
    .options(keyspace="system", table="local")
    .load()
    .select("cluster_name", "release_version")
    .show(truncate=False)
)
print("Spark-Cassandra connection works")

In [ ]:
from pymongo import MongoClient

mongo_client = MongoClient(MONGODB_URI, serverSelectionTimeoutMS=15000)
mongo_client.admin.command("ping")                 # raises an error if the connection fails
mongo_db = mongo_client["ind320"]
print("Connected to MongoDB. Databases visible:", mongo_client.list_database_names())

# CRUD test on a temporary collection
test_col = mongo_db["_connection_test"]
test_col.delete_many({})
test_col.insert_one({"name": "test", "value": 1})                                   # Create
print("Read:   ", test_col.find_one({"name": "test"}, {"_id": 0}))                   # Read
test_col.update_one({"name": "test"}, {"$set": {"value": 2}})                       # Update
print("Updated:", test_col.find_one({"name": "test"}, {"_id": 0}))
test_col.delete_many({})                                                            # Delete
print("Documents left after delete:", test_col.count_documents({}))

The reservoir data is now fetched directly from the NVE API instead of `data/reservoirs.csv`.

In [ ]:
NVE_BASE = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk"

# Download all public reservoir statistics
resp = requests.get(f"{NVE_BASE}/HentOffentligData", timeout=60)
resp.raise_for_status()
nve_raw = pd.DataFrame(resp.json())
print("Rows downloaded:", len(nve_raw))
print("Original (Norwegian) columns:", nve_raw.columns.tolist())

# Rename to the same English names as in part 1
nve_all = nve_raw.rename(columns={
    "dato_Id":                  "date",
    "omrType":                  "area_type",
    "omrnr":                    "area_nr",
    "iso_aar":                  "year",
    "iso_uke":                  "week",
    "fyllingsgrad":             "fill_level",
    "kapasitet_TWh":            "capacity_TWh",
    "fylling_TWh":              "stored_energy_TWh",
    "neste_Publiseringsdato":   "next_publish_date",
    "fyllingsgrad_forrige_uke": "fill_level_previous_week",
    "endring_fyllingsgrad":     "change_in_fill_level",
})

# Convert the text dates to datetime and sort chronologically
nve_all["date"] = pd.to_datetime(nve_all["date"], errors="coerce")
if "next_publish_date" in nve_all.columns:
    nve_all["next_publish_date"] = pd.to_datetime(nve_all["next_publish_date"], errors="coerce")
nve_all = nve_all.sort_values(["area_type", "area_nr", "date"]).reset_index(drop=True)

nve_all.info()
nve_all.head()

In [ ]:
# One row per area: how many weekly observations and which period it covers
areas = (
    nve_all.groupby(["area_type", "area_nr"])
    .agg(first_date=("date", "min"), last_date=("date", "max"), n_weeks=("date", "count"))
    .reset_index()
)

# Summary per area type
area_summary = (
    areas.groupby("area_type")
    .agg(
        n_areas=("area_nr", "nunique"),
        area_numbers=("area_nr", lambda s: sorted(s.tolist())),
        first_date=("first_date", "min"),
        last_date=("last_date", "max"),
    )
)
display(area_summary)

print("Full list of areas:")
display(areas)

In [ ]:
# Select one area from the API data
AREA_TYPE = "NO"                       # one of EL, NO, VASS
AREA_NR = None                         # None = first area number of the chosen type

subset = nve_all[nve_all["area_type"] == AREA_TYPE]
assert len(subset) > 0, f"No data for area type {AREA_TYPE}"
if AREA_NR is None:
    AREA_NR = sorted(subset["area_nr"].unique())[0]

df = subset[subset["area_nr"] == AREA_NR].sort_values("date").reset_index(drop=True)
print(f"Area type {AREA_TYPE}, area number {AREA_NR}: {len(df)} weekly rows, "
      f"{df['date'].min().date()} to {df['date'].max().date()}")

# Numerical reservoir columns (same as part 1)
plot_columns = [
    "fill_level",
    "capacity_TWh",
    "stored_energy_TWh",
    "fill_level_previous_week",
    "change_in_fill_level",
]

display(df[plot_columns].describe().T)

In [ ]:
# Plot each numerical column separately
for column in plot_columns:
    plt.figure(figsize=(10, 4))
    plt.plot(df["date"], df[column], linewidth=0.8, color="steelblue")
    plt.title(f"{column} - {AREA_TYPE} {AREA_NR}")
    plt.xlabel("Date")
    plt.ylabel(column)
    plt.grid(True)
    plt.show()

In [ ]:
# Plot all variables together, min-max normalised to 0-1 so the different units are comparable
df_norm = df.copy()
for col in plot_columns:
    mn, mx = df[col].min(), df[col].max()
    # Guard against division by zero if a column is constant
    df_norm[col] = (df[col] - mn) / (mx - mn) if mx != mn else 0.0

plt.figure(figsize=(14, 6))
for col in plot_columns:
    plt.plot(df["date"], df_norm[col], linewidth=1.2, alpha=0.8, label=col)

plt.title(f"All reservoir variables together (normalized 0-1) - {AREA_TYPE} {AREA_NR}")
plt.xlabel("Date")
plt.ylabel("Normalized value (0-1)")
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

Physical cross-border flows are retrieved from the [ENTSO-E Transparency Platform](https://transparency.entsoe.eu/) using document type `A11`.

In [ ]:
# Bidding zones: code -> (EIC code, country)
ZONES = {
    "NO1":   ("10YNO-1--------2",   "Norway"),
    "NO2":   ("10YNO-2--------T",   "Norway"),
    "NO3":   ("10YNO-3--------J",   "Norway"),
    "NO4":   ("10YNO-4--------9",   "Norway"),
    "SE1":   ("10Y1001A1001A44P",   "Sweden"),
    "SE2":   ("10Y1001A1001A45N",   "Sweden"),
    "SE3":   ("10Y1001A1001A46L",   "Sweden"),
    "FI":    ("10YFI-1--------U",   "Finland"),
    "DK1":   ("10YDK-1--------W",   "Denmark"),
    "DE-LU": ("10Y1001A1001A82H",   "Germany"),
    "GB":    ("10YGB----------A",   "Great Britain"),
    "NL":    ("10YNL----------L",   "Netherlands"),
}

# Norwegian zone first, foreign zone second (same list as the Streamlit app)
CORRIDORS = [
    "NO1 - SE3", "NO2 - DE-LU", "NO2 - DK1", "NO2 - GB", "NO2 - NL",
    "NO3 - SE2", "NO4 - FI", "NO4 - SE1", "NO4 - SE2",
]

# The last two complete calendar years (today is in 2026)
YEARS = [2024, 2025]

ENTSOE_API = "https://web-api.tp.entsoe.eu/api"


def corridor_title(corridor):
    # 'NO2 - DK1' -> 'Norway - Denmark'
    no_zone, foreign_zone = [z.strip() for z in corridor.split(" - ")]
    return f"{ZONES[no_zone][1]} - {ZONES[foreign_zone][1]}"

In [ ]:
def resolution_to_minutes(res):
    # ISO 8601 duration used by ENTSO-E, e.g. PT15M, PT30M, PT60M, PT1H
    m = re.fullmatch(r"PT(\d+)([MH])", res.strip())
    if not m:
        raise ValueError(f"Unknown resolution: {res}")
    n, unit = int(m.group(1)), m.group(2)
    return n * 60 if unit == "H" else n


def parse_entsoe_xml(xml_text):
    # Parse an ENTSO-E physical flow XML document into a DataFrame
    # Columns: ts (UTC, timezone-naive), flow_mw, resolution_min.
    # An empty DataFrame is returned for 'no data' (acknowledgement) documents.
    cols = ["ts", "flow_mw", "resolution_min"]
    root = ET.fromstring(xml_text)
    ns = root.tag[1:root.tag.index("}")] if root.tag.startswith("{") else ""
    q = (lambda name: f"{{{ns}}}{name}") if ns else (lambda name: name)

    rows = []
    for series in root.iter(q("TimeSeries")):
        curve_type = series.findtext(q("curveType"))          # A01 = every point, A03 = only changes
        for period in series.findall(q("Period")):
            ti = period.find(q("timeInterval"))
            start = datetime.fromisoformat(ti.findtext(q("start")).replace("Z", "+00:00"))
            end = datetime.fromisoformat(ti.findtext(q("end")).replace("Z", "+00:00"))
            res = resolution_to_minutes(period.findtext(q("resolution")))
            step = timedelta(minutes=res)
            n_slots = int((end - start) / step)

            # position -> quantity as given in the XML
            values = {
                int(pt.findtext(q("position"))): float(pt.findtext(q("quantity")))
                for pt in period.findall(q("Point"))
            }

            last = None
            for pos in range(1, n_slots + 1):
                if pos in values:
                    last = values[pos]
                elif curve_type != "A03":
                    continue                                   # a real gap: leave it missing
                if last is None:
                    continue
                rows.append((start + (pos - 1) * step, last, res))

    if not rows:
        return pd.DataFrame(columns=cols)
    out = pd.DataFrame(rows, columns=cols)
    out["ts"] = pd.to_datetime(out["ts"]).dt.tz_convert("UTC").dt.tz_localize(None)   # naive UTC
    return out.drop_duplicates("ts").sort_values("ts").reset_index(drop=True)


def fetch_flow(out_eic, in_eic, year, retries=3):
    # Download one direction of one connection for one calendar year (UTC)
    params = {
        "securityToken": ENTSOE_TOKEN,
        "documentType":  "A11",
        "out_Domain":    out_eic,
        "in_Domain":     in_eic,
        "periodStart":   f"{year}01010000",
        "periodEnd":     f"{year + 1}01010000",
    }
    for attempt in range(1, retries + 1):
        try:
            r = requests.get(ENTSOE_API, params=params, timeout=90)
        except requests.RequestException as exc:
            if attempt == retries:
                return pd.DataFrame(), f"request failed: {exc}"
            time.sleep(2 * attempt)
            continue
        if r.status_code == 401:
            raise RuntimeError("ENTSO-E rejected the security token (HTTP 401)")
        if r.status_code in (429, 500, 502, 503, 504) and attempt < retries:
            time.sleep(3 * attempt)                            # back off and retry
            continue
        if r.status_code != 200:
            return pd.DataFrame(), f"HTTP {r.status_code}"
        df = parse_entsoe_xml(r.text)
        return df, ("ok" if len(df) else "no data returned")
    return pd.DataFrame(), "failed"

In [ ]:
# Download every connection, both directions, for both years
frames, status_rows = [], []

for corridor in CORRIDORS:
    no_zone, foreign_zone = [z.strip() for z in corridor.split(" - ")]
    no_eic, foreign_eic = ZONES[no_zone][0], ZONES[foreign_zone][0]

    directions = {
        "Into Norway":   (foreign_eic, no_eic, foreign_zone, no_zone),
        "Out of Norway": (no_eic, foreign_eic, no_zone, foreign_zone),
    }
    for direction, (out_eic, in_eic, out_zone, in_zone) in directions.items():
        for year in YEARS:
            df_flow, status = fetch_flow(out_eic, in_eic, year)
            status_rows.append({"corridor": corridor, "direction": direction, "year": year,
                                "rows": len(df_flow), "status": status})
            if len(df_flow):
                df_flow = df_flow.assign(corridor=corridor, direction=direction,
                                         out_zone=out_zone, in_zone=in_zone)
                frames.append(df_flow)
            time.sleep(0.3)                                    # be polite to the API

download_status = pd.DataFrame(status_rows)
display(download_status)

flows = pd.concat(frames, ignore_index=True)
flows["year"] = flows["ts"].dt.year.astype(int)               # UTC year, also used as partition key
print("Total rows downloaded:", len(flows))
flows.head()

In [ ]:
# Quick quality check: resolution used and period covered per connection and direction
quality = (
    flows.groupby(["corridor", "direction"])
    .agg(rows=("ts", "count"), first_ts=("ts", "min"), last_ts=("ts", "max"),
         resolutions_min=("resolution_min", lambda s: sorted(s.unique().tolist())),
         missing_flow=("flow_mw", lambda s: int(s.isna().sum())))
    .reset_index()
)
display(quality)

In [ ]:
# Explicit schema so the Spark types match the Cassandra table
cass_schema = T.StructType([
    T.StructField("corridor",       T.StringType(),    False),
    T.StructField("direction",      T.StringType(),    False),
    T.StructField("year",           T.IntegerType(),   False),
    T.StructField("ts",             T.TimestampType(), False),
    T.StructField("flow_mw",        T.DoubleType(),    True),
    T.StructField("resolution_min", T.IntegerType(),   True),
    T.StructField("out_zone",       T.StringType(),    True),
    T.StructField("in_zone",        T.StringType(),    True),
])

# Build plain Python rows with UTC-aware datetimes
rows = [
    (c, d, int(y), t.to_pydatetime().replace(tzinfo=timezone.utc), float(v), int(r), o, i)
    for c, d, y, t, v, r, o, i in zip(
        flows["corridor"], flows["direction"], flows["year"], flows["ts"],
        flows["flow_mw"], flows["resolution_min"], flows["out_zone"], flows["in_zone"])
]
sdf_flows = spark.createDataFrame(rows, schema=cass_schema)
print("Spark DataFrame rows to insert:", sdf_flows.count())

# Write to Cassandra through Spark
(
    sdf_flows.write
    .format("org.apache.spark.sql.cassandra")
    .mode("append")
    .options(keyspace=KEYSPACE, table=TABLE)
    .save()
)
print("Write to Cassandra finished")

In [ ]:
# Verify: count the rows that now exist in Cassandra
cass_df = (
    spark.read.format("org.apache.spark.sql.cassandra")
    .options(keyspace=KEYSPACE, table=TABLE)
    .load()
)
print("Rows in Cassandra:", cass_df.count())

In [ ]:
# Keep only the columns needed and add the energy of each interval (MW x hours = MWh)
base = (
    cass_df
    .select("corridor", "direction", "year", "ts", "flow_mw", "resolution_min")
    .where(F.col("flow_mw").isNotNull())
    .withColumn("energy_mwh", F.col("flow_mw") * F.col("resolution_min") / 60.0)
)

# 1) Yearly totals per connection and direction
yearly_sdf = (
    base.groupBy("corridor", "direction", "year")
    .agg(
        (F.sum("energy_mwh") / 1e6).alias("energy_twh"),
        F.avg("flow_mw").alias("avg_mw"),
        F.max("flow_mw").alias("max_mw"),
        F.count("*").alias("n_obs"),
    )
    .orderBy("corridor", "direction", "year")
)

# 2) Hourly series per connection and direction
hourly_sdf = (
    base.withColumn("hour", F.date_trunc("hour", F.col("ts")))
    .groupBy("corridor", "direction", "year", "hour")
    .agg(F.avg("flow_mw").alias("flow_mw"))
    .withColumn("month", F.month("hour"))
    .withColumn("ts", F.date_format("hour", "yyyy-MM-dd HH:mm:ss"))   # text in UTC (session time zone)
    .select("corridor", "direction", "year", "month", "ts", "flow_mw")
    .orderBy("corridor", "direction", "ts")
)

yearly_sdf.show(40, truncate=False)

In [ ]:
# Collect the (small) Spark results into pandas for plotting and for MongoDB
yearly_pd = pd.DataFrame([r.asDict() for r in yearly_sdf.collect()])
hourly_pd = pd.DataFrame([r.asDict() for r in hourly_sdf.collect()])
hourly_pd["ts"] = pd.to_datetime(hourly_pd["ts"])            # naive UTC timestamps

print("Yearly rows:", len(yearly_pd), "| Hourly rows:", len(hourly_pd))
hourly_pd.head()

### Pie chart: total yearly transfer for one connection

Each piece of the pie is the amount of energy that flowed *into* or *out of* Norway through the chosen connection during the chosen year.

In [ ]:
# Yearly energy per direction for the chosen connection and year (from the Spark extract)
sel = yearly_pd[(yearly_pd["corridor"] == CHOSEN_CORRIDOR) & (yearly_pd["year"] == CHOSEN_YEAR)]
totals = sel.set_index("direction")["energy_twh"]
labels = [d for d in ["Into Norway", "Out of Norway"] if d in totals.index]
values = [totals[d] for d in labels]

if sum(values) == 0 or not values:
    print("No data for this connection/year")
else:
    fig, ax = plt.subplots(figsize=(7, 5))
    wedges, texts, autotexts = ax.pie(
        values,
        colors=[PIE_COLORS[d] for d in labels],
        autopct=lambda p: "",                 # labels are written below with TWh and %
        startangle=90, counterclock=False, pctdistance=0.62,
        wedgeprops={"edgecolor": "white", "linewidth": 1.5},
        textprops={"color": "white", "fontsize": 10, "ha": "center"},
    )
    for at, lab, val in zip(autotexts, labels, values):
        at.set_text(f"{lab}\n{val:.2f} TWh\n{val / sum(values) * 100:.1f}%")
    ax.legend(wedges, labels, title="Direction", loc="center left",
              bbox_to_anchor=(1.0, 0.5), frameon=False)
    ax.set_title(f"Yearly electricity transfer - {corridor_title(CHOSEN_CORRIDOR)} ({CHOSEN_YEAR})",
                 loc="left", fontsize=11)
    plt.show()

### Line plot: first month of the year

January of the chosen year. The first plot shows the chosen connection (one line per direction). The second plot shows **all connections**, with one line for each connection *and* direction (colour = connection, solid line = into Norway, dashed line = out of Norway).

In [ ]:
# Line plot 1: the chosen connection, January, one line per direction
jan = hourly_pd[(hourly_pd["year"] == CHOSEN_YEAR) & (hourly_pd["month"] == 1)]
one = jan[jan["corridor"] == CHOSEN_CORRIDOR]

fig, ax = plt.subplots(figsize=(11, 4.8))
for direction in ["Into Norway", "Out of Norway"]:
    d = one[one["direction"] == direction].sort_values("ts")
    if len(d):
        ax.plot(d["ts"], d["flow_mw"], color=LINE_COLORS[direction], linewidth=1.2, label=direction)

ax.set_title(f"Electricity flow, January {CHOSEN_YEAR} - {corridor_title(CHOSEN_CORRIDOR)}")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Flow (MW)")
ax.grid(True, alpha=0.5)
ax.legend()
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
# Line plot 2: all connections, January, separate line for each connection and direction
fig, ax = plt.subplots(figsize=(13, 6))
palette = plt.cm.tab10(np.linspace(0, 1, 10))
corridor_color = {c: palette[i % 10] for i, c in enumerate(CORRIDORS)}

for corridor in CORRIDORS:
    for direction, linestyle in [("Into Norway", "-"), ("Out of Norway", "--")]:
        d = jan[(jan["corridor"] == corridor) & (jan["direction"] == direction)].sort_values("ts")
        if len(d):
            ax.plot(d["ts"], d["flow_mw"], color=corridor_color[corridor],
                    linestyle=linestyle, linewidth=1.0, label=f"{corridor} ({direction})")

ax.set_title(f"Electricity flow, January {CHOSEN_YEAR} - all connections")
ax.set_xlabel("Date (UTC)")
ax.set_ylabel("Flow (MW)")
ax.grid(True, alpha=0.4)
ax.legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=8, frameon=False)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
def to_documents(df, ts_col=None):
    # Convert a DataFrame to a list of MongoDB documents with native Python types
    docs = df.to_dict("records")
    for doc in docs:
        for key, val in doc.items():
            if isinstance(val, np.integer):
                doc[key] = int(val)                               # numpy int -> Python int
            elif isinstance(val, np.floating):
                doc[key] = float(val)                             # numpy float -> Python float
        if ts_col:
            doc[ts_col] = doc[ts_col].to_pydatetime()             # naive datetime = UTC in MongoDB
    return docs


# Yearly totals (small)
yearly_docs = to_documents(yearly_pd.round({"energy_twh": 4, "avg_mw": 2, "max_mw": 2}))
col_yearly = mongo_db["entsoe_yearly"]
col_yearly.delete_many({})
col_yearly.insert_many(yearly_docs)
col_yearly.create_index([("corridor", 1), ("year", 1), ("direction", 1)])
print("entsoe_yearly documents:", col_yearly.count_documents({}))

# Hourly series (larger) - inserted in batches
hourly_docs = to_documents(hourly_pd.round({"flow_mw": 2}), ts_col="ts")
col_hourly = mongo_db["entsoe_hourly"]
col_hourly.delete_many({})
BATCH = 20000
for i in range(0, len(hourly_docs), BATCH):
    col_hourly.insert_many(hourly_docs[i:i + BATCH], ordered=False)
col_hourly.create_index([("corridor", 1), ("year", 1), ("month", 1), ("direction", 1)])
col_hourly.create_index([("ts", 1)])
print("entsoe_hourly documents:", col_hourly.count_documents({}))

In [ ]:
# Verify the insert: read some documents back and compare with the Spark result
print("Sample yearly document:", col_yearly.find_one({}, {"_id": 0}))
print("Sample hourly document:", col_hourly.find_one({}, {"_id": 0}))

assert col_yearly.count_documents({}) == len(yearly_pd), "Yearly count differs from Spark result"
assert col_hourly.count_documents({}) == len(hourly_pd), "Hourly count differs from Spark result"
print("MongoDB contents match the Spark-extracted data")

In [ ]:
# Close all connections
spark.stop()
cluster.shutdown()
mongo_client.close()
print("Connections closed")

**Log:**
It was challanging, becaouse of issues with the installment and setup part. 
most issues with spark, pyspark version and configuring hadoop, spark worked together with cassandra.
Took long time to get acssec to Entsoe API Token. 


**AI Usage:**
used AI for setup of Docker, Cassandra, MongoDB, Spark and APIs. 
notebook i used AI in many parts, Got Spark errors alot. 